# Stream B — `clip_features.ipynb`
**Purpose:** Load pretrained `merged_balanced_vitb32/best.ckpt`, extract penultimate-layer features
for NewsCLIPpings (train/val/test), VERITE, and MMFakeBench, and save to `cache/stream_b_features/`.

**Key rules from GRAND_PLAN:**
- NC embedding keys are **int** (not str)
- NC uses composite key `f"{id}_{int(falsified)}"` for pair-level caches
- places_resnet50 has **no train.pkl** — skip for train
- VERITE images: use `data/verite/image-text-verification/VERITE/images/`
- MMFakeBench root: `data/mmfakebench/MMFakeBench_test/`
- Backbone: `data/newsclippings/models/vitb32/merged_balanced_vitb32/best.ckpt`

## 0. Environment Check

In [9]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch version: {torch.__version__}")

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM: 8.6 GB
PyTorch version: 2.6.0+cu124


## 1. Imports & Paths

In [10]:
import os, sys, json, pickle
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import clip  # openai/CLIP

# ── Resolve project root ──────────────────────────────────────────────────────
# Works whether run from notebooks/03_stream_b/ or the project root
try:
    ROOT = Path(__file__).resolve().parents[2]
except NameError:
    ROOT = Path(os.getcwd())
    if ROOT.name in ('03_stream_b', 'notebooks'):
        ROOT = ROOT.parents[1] if ROOT.name == '03_stream_b' else ROOT.parent
    # Walk up until we find src/utils/paths.py as a sanity check
    while not (ROOT / 'src' / 'utils' / 'paths.py').exists():
        ROOT = ROOT.parent
        if ROOT == ROOT.parent:   # filesystem root — give up
            ROOT = Path(os.getcwd())
            break

sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

# ── Canonical path constants (mirrors src/utils/paths.py) ────────────────────
NEWSCLIP_ROOT = ROOT / 'data' / 'newsclippings'
CACHE_DIR     = ROOT / 'cache'
CLIP_EMB_DIR  = CACHE_DIR / 'clip_embeddings'
STREAM_B_DIR  = CACHE_DIR / 'stream_b_features'
STREAM_B_DIR.mkdir(parents=True, exist_ok=True)

# Pretrained backbone
CKPT_PATH = NEWSCLIP_ROOT / 'models' / 'vitb32' / 'merged_balanced_vitb32' / 'test.ckpt'

# Dataset JSON paths
NC_SPLITS = {
    'train': NEWSCLIP_ROOT / 'train.json',   # ROOT-level, 71 072 pairs
    'val':   NEWSCLIP_ROOT / 'data' / 'merged_balanced' / 'val.json',
    'test':  NEWSCLIP_ROOT / 'test.json',    # ROOT-level, 7 264 pairs
}
NC_META_DIR = NEWSCLIP_ROOT / 'metadata'

VERITE_CSV  = ROOT / 'data' / 'verite' / 'image-text-verification' / 'VERITE' / 'VERITE.csv'
VERITE_IMGS = ROOT / 'data' / 'verite' / 'image-text-verification' / 'VERITE' / 'images'

MMFAKE_JSON = ROOT / 'data' / 'mmfakebench' / 'MMFakeBench_test.json'
MMFAKE_ROOT = ROOT / 'data' / 'mmfakebench' / 'MMFakeBench_test'

# Verify critical paths exist
for name, p in [('CKPT', CKPT_PATH), ('NC train', NC_SPLITS['train']),
                ('NC test',  NC_SPLITS['test']),
                ('VERITE csv', VERITE_CSV), ('VERITE imgs', VERITE_IMGS),
                ('MMFake json', MMFAKE_JSON)]:
    status = '✅' if p.exists() else '❌ MISSING'
    print(f"{status}  {name}: {p}")

Project root: f:\GitHub\eve-rag
✅  CKPT: f:\GitHub\eve-rag\data\newsclippings\models\vitb32\merged_balanced_vitb32\test.ckpt
✅  NC train: f:\GitHub\eve-rag\data\newsclippings\train.json
✅  NC test: f:\GitHub\eve-rag\data\newsclippings\test.json
✅  VERITE csv: f:\GitHub\eve-rag\data\verite\image-text-verification\VERITE\VERITE.csv
✅  VERITE imgs: f:\GitHub\eve-rag\data\verite\image-text-verification\VERITE\images
✅  MMFake json: f:\GitHub\eve-rag\data\mmfakebench\MMFakeBench_test.json


## 2. Load Pre-extracted CLIP Embeddings

NC images are **not** on disk — we use the pre-extracted `cache/clip_embeddings/` pickles.
Keys are **int** image_ids.

In [11]:
def load_pkl(path):
    """Load a pickle file and return its contents."""
    with open(path, 'rb') as f:
        return pickle.load(f)

print("Loading CLIP image embeddings …")
clip_img = {
    'train': load_pkl(CLIP_EMB_DIR / 'clip_image_embeddings' / 'train.pkl'),
    'val':   load_pkl(CLIP_EMB_DIR / 'clip_image_embeddings' / 'val.pkl'),
    'test':  load_pkl(CLIP_EMB_DIR / 'clip_image_embeddings' / 'test.pkl'),
}
print("Loading CLIP text embeddings …")
clip_txt = {
    'train': load_pkl(CLIP_EMB_DIR / 'clip_text_embeddings' / 'train.pkl'),
    'val':   load_pkl(CLIP_EMB_DIR / 'clip_text_embeddings' / 'val.pkl'),
    'test':  load_pkl(CLIP_EMB_DIR / 'clip_text_embeddings' / 'test.pkl'),
}
print("Loading FaceNet embeddings …")
facenet = {
    'train': load_pkl(CLIP_EMB_DIR / 'facenet_embeddings' / 'train.pkl'),
    'val':   load_pkl(CLIP_EMB_DIR / 'facenet_embeddings' / 'val.pkl'),
    'test':  load_pkl(CLIP_EMB_DIR / 'facenet_embeddings' / 'test.pkl'),
}
print("Loading SBERT embeddings …")
sbert = {
    'train': load_pkl(CLIP_EMB_DIR / 'sbert_embeddings' / 'train.pkl'),
    'val':   load_pkl(CLIP_EMB_DIR / 'sbert_embeddings' / 'val.pkl'),
    'test':  load_pkl(CLIP_EMB_DIR / 'sbert_embeddings' / 'test.pkl'),
}
# places_resnet50: NO train.pkl — val and test only
print("Loading Places-ResNet50 embeddings (val + test only) …")
places = {
    'val':  load_pkl(CLIP_EMB_DIR / 'places_resnet50' / 'val.pkl'),
    'test': load_pkl(CLIP_EMB_DIR / 'places_resnet50' / 'test.pkl'),
}

# Quick sanity: check key type (must be int)
for name, d in [('clip_img[test]', clip_img['test']),
                ('clip_txt[test]', clip_txt['test'])]:
    sample_key = next(iter(d))
    print(f"{name}: {len(d)} entries, sample key type = {type(sample_key).__name__} → "
          f"{'✅ int' if isinstance(sample_key, (int, np.integer)) else '❌ NOT int — fix needed'}")

Loading CLIP image embeddings …
Loading CLIP text embeddings …
Loading FaceNet embeddings …
Loading SBERT embeddings …
Loading Places-ResNet50 embeddings (val + test only) …
clip_img[test]: 50000 entries, sample key type = int → ✅ int
clip_txt[test]: 40915 entries, sample key type = int → ✅ int


## 3. Load Pretrained ViT-B/32 Backbone

The checkpoint was trained with the NewsCLIPpings library. We load it and hook the
penultimate layer to extract 512-d image and text representations.

In [12]:
import yaml

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ── Load config ───────────────────────────────────────────────────────────────
CONFIG_PATH = CKPT_PATH.parent / 'config.yaml'
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        config = yaml.safe_load(f)
    print("Config loaded:", json.dumps(config, indent=2, default=str))
else:
    config = {}
    print("⚠️  config.yaml not found — proceeding with defaults")

# ── Load vanilla CLIP ViT-B/32 ────────────────────────────────────────────────
# The checkpoint may be a full state_dict or a Lightning checkpoint.
# We load the CLIP backbone first, then overlay the checkpoint weights.
clip_model, clip_preprocess = clip.load('ViT-B/32', device=DEVICE)
clip_model.eval()
print("Vanilla CLIP ViT-B/32 loaded.")

# ── Attempt to load checkpoint weights ───────────────────────────────────────
ckpt_raw = torch.load(CKPT_PATH, map_location=DEVICE)

# Detect format: Lightning checkpoints have 'state_dict' key
if isinstance(ckpt_raw, dict) and 'state_dict' in ckpt_raw:
    state_dict = ckpt_raw['state_dict']
    print(f"Lightning checkpoint detected. Keys (first 5): {list(state_dict.keys())[:5]}")
elif isinstance(ckpt_raw, dict):
    state_dict = ckpt_raw
    print(f"Raw state_dict detected. Keys (first 5): {list(state_dict.keys())[:5]}")
else:
    state_dict = None
    print("⚠️  Checkpoint format unrecognised — using vanilla CLIP weights only")

# Strip any 'model.' or 'clip_model.' prefix that Lightning adds
if state_dict is not None:
    def strip_prefix(sd, prefix):
        return {k[len(prefix):]: v for k, v in sd.items() if k.startswith(prefix)}
    
    # Try common prefixes
    for prefix in ('model.clip_model.', 'clip_model.', 'model.', ''):
        stripped = strip_prefix(state_dict, prefix) if prefix else state_dict
        # Check if any key matches clip_model
        clip_keys = set(clip_model.state_dict().keys())
        overlap = len(set(stripped.keys()) & clip_keys)
        if overlap > 0:
            print(f"Prefix '{prefix}': {overlap}/{len(clip_keys)} keys match CLIP model")
            try:
                missing, unexpected = clip_model.load_state_dict(stripped, strict=False)
                print(f"Loaded with strict=False — missing: {len(missing)}, unexpected: {len(unexpected)}")
                break
            except Exception as e:
                print(f"Failed with prefix '{prefix}': {e}")
    else:
        print("⚠️  Could not match checkpoint keys — using vanilla CLIP weights")

clip_model.eval()
print("Backbone ready.")

Using device: cuda
Config loaded: {
  "config_version": 1.0,
  "training": {
    "trainer": "mmf",
    "seed": 2,
    "experiment_name": "run",
    "max_updates": 44000,
    "max_epochs": null,
    "log_interval": 100,
    "logger_level": "info",
    "log_format": "simple",
    "log_detailed_config": false,
    "should_not_log": false,
    "colored_logs": true,
    "tensorboard": true,
    "batch_size": 32,
    "update_frequency": 1,
    "num_workers": 4,
    "fast_read": false,
    "dataset_size_proportional_sampling": true,
    "pin_memory": false,
    "checkpoint_interval": 1000,
    "evaluation_interval": 1000,
    "clip_gradients": false,
    "clip_norm_mode": "all",
    "early_stop": {
      "enabled": false,
      "patience": 4000,
      "criteria": "total_loss",
      "minimize": true
    },
    "lr_scheduler": true,
    "lr_steps": [],
    "lr_ratio": 0.1,
    "use_warmup": false,
    "warmup_factor": 0.2,
    "warmup_iterations": 1000,
    "device": "cuda",
    "local_rank": 

## 4. Feature Extraction Helpers

In [13]:
import torch.nn.functional as F

def cosine_sim(a, b):
    """Cosine similarity between two L2-normalised numpy vectors."""
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    return float(np.dot(a, b))

def to_numpy(x):
    """Safely convert tensor or array to float32 numpy."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().float().numpy()
    return np.array(x, dtype=np.float32)

def get_embedding(emb_dict, key):
    """Retrieve embedding by int key; return None if missing."""
    key = int(key)  # always int
    val = emb_dict.get(key, None)
    if val is None:
        return None
    return to_numpy(val)

def l2_norm(v):
    """L2-normalise a numpy vector."""
    n = np.linalg.norm(v)
    return v / (n + 1e-8)

## 5. Build Stream B Feature Matrix — NewsCLIPpings

For each pair we assemble:
- `clip_img_emb` (512-d) — ViT-B/32 image embedding
- `clip_txt_emb` (512-d) — ViT-B/32 text embedding
- `clip_sim` (scalar) — cosine similarity
- `sbert_emb` (768-d) — SBERT text embedding
- `facenet_emb` (512-d) — FaceNet face embedding (zeros if no face)
- `places_emb` (365-d) — Places-ResNet50 (val/test only; zeros for train)
- `label` (0/1)
- `pair_key` — composite `f"{id}_{int(falsified)}"`

In [14]:
def infer_emb_dim(emb_dict):
    """Get embedding dimension from first entry."""
    v = next(iter(emb_dict.values()))
    return to_numpy(v).shape[-1]

# Embedding dimensions
DIM_CLIP  = infer_emb_dim(clip_img['test'])   # 512
DIM_SBERT = infer_emb_dim(sbert['test'])       # 768
DIM_FACE  = infer_emb_dim(facenet['test'])     # 512
DIM_PLACE = infer_emb_dim(places['test'])      # 365
print(f"Dims — CLIP: {DIM_CLIP}, SBERT: {DIM_SBERT}, FaceNet: {DIM_FACE}, Places: {DIM_PLACE}")

Dims — CLIP: 512, SBERT: 768, FaceNet: 512, Places: 2048


In [15]:
def build_nc_features(split):
    """
    Build Stream B feature dict for a NewsCLIPpings split.
    Returns a dict keyed by composite pair_key.
    """
    json_path = NC_SPLITS[split]
    with open(json_path) as f:
        data = json.load(f)
    # ROOT-level JSONs: list of {id, image_id, falsified, ...}
    # merged_balanced/val.json: same schema
    pairs = data if isinstance(data, list) else data.get('annotations', data)

    ci = clip_img[split]
    ct = clip_txt[split]
    sb = sbert[split]
    fn = facenet[split]
    pl = places.get(split, None)   # None for 'train'

    results = {}
    n_missing = 0

    for row in tqdm(pairs, desc=f"NC {split}", leave=False):
        image_id  = int(row['image_id'])
        pair_id   = int(row['id'])
        falsified = int(row['falsified'])
        label     = falsified          # 1 = OOC, 0 = genuine
        pair_key  = f"{pair_id}_{falsified}"  # composite key

        img_e = get_embedding(ci, image_id)
        txt_e = get_embedding(ct, pair_id)

        if img_e is None or txt_e is None:
            n_missing += 1
            continue

        clip_s = cosine_sim(img_e, txt_e)

        sbert_e = get_embedding(sb, pair_id)
        if sbert_e is None:
            sbert_e = np.zeros(DIM_SBERT, dtype=np.float32)

        face_e = get_embedding(fn, image_id)
        if face_e is None:
            face_e = np.zeros(DIM_FACE, dtype=np.float32)

        if pl is not None:
            place_e = get_embedding(pl, image_id)
            if place_e is None:
                place_e = np.zeros(DIM_PLACE, dtype=np.float32)
        else:
            place_e = np.zeros(DIM_PLACE, dtype=np.float32)

        results[pair_key] = {
            'pair_key':   pair_key,
            'pair_id':    pair_id,
            'image_id':   image_id,
            'label':      label,
            'clip_img_emb': l2_norm(img_e).astype(np.float32),
            'clip_txt_emb': l2_norm(txt_e).astype(np.float32),
            'clip_sim':   np.float32(clip_s),
            'sbert_emb':  l2_norm(sbert_e).astype(np.float32),
            'facenet_emb': l2_norm(face_e).astype(np.float32),
            'places_emb': l2_norm(place_e).astype(np.float32),
        }

    print(f"  NC {split}: {len(results)} pairs built, {n_missing} skipped (missing emb)")
    return results

nc_features = {}
for split in ('train', 'val', 'test'):
    nc_features[split] = build_nc_features(split)

NC train:   0%|          | 0/71072 [00:00<?, ?it/s]

  NC train: 71072 pairs built, 0 skipped (missing emb)


NC val:   0%|          | 0/7024 [00:00<?, ?it/s]

  NC val: 7024 pairs built, 0 skipped (missing emb)


NC test:   0%|          | 0/7264 [00:00<?, ?it/s]

  NC test: 7264 pairs built, 0 skipped (missing emb)


## 6. Build Stream B Feature Matrix — VERITE

In [16]:
from PIL import Image

# VERITE ViT-L/14 pre-extracted embeddings (768-d) — also available
VERITE_VITL_IMG = ROOT / 'data' / 'verite' / 'image-text-verification' / 'VERITE' / 'VERITE_clip_image_embeddings_ViTL14.npy'
# Fallback: cache/clip_embeddings/verite/
VERITE_CACHE_IMG = CLIP_EMB_DIR / 'verite' / 'image_embeddings.npy'
VERITE_CACHE_TXT = CLIP_EMB_DIR / 'verite' / 'text_embeddings.npy'

# Load VERITE CSV
df_verite = pd.read_csv(VERITE_CSV)
print(f"VERITE: {len(df_verite)} rows. Columns: {list(df_verite.columns)}")
print(df_verite.head(3))

VERITE: 1001 rows. Columns: ['Unnamed: 0', 'caption', 'image_path', 'label']
   Unnamed: 0                                            caption  \
0           0  Photograph shows a family that was taken away ...   
1           1  Photograph shows Chinese officials in white pr...   
2           2  Photograph shows a family that was taken away ...   

           image_path           label  
0   images/true_0.jpg            true  
1   images/true_0.jpg    miscaptioned  
2  images/false_0.jpg  out-of-context  


In [17]:
# ── Load VERITE embeddings ────────────────────────────────────────────────────
# Prefer cache/clip_embeddings/verite/ (ViT-B/32, consistent with NC)
# Fall back to ViT-L/14 if cache not available

verite_img_embs = None
verite_txt_embs = None
verite_emb_dim  = None

if VERITE_CACHE_IMG.exists() and VERITE_CACHE_TXT.exists():
    verite_img_embs = np.load(VERITE_CACHE_IMG)  # shape (N, D)
    verite_txt_embs = np.load(VERITE_CACHE_TXT)
    verite_emb_dim  = verite_img_embs.shape[1]
    print(f"Loaded VERITE ViT-B/32 embeddings from cache: img {verite_img_embs.shape}, txt {verite_txt_embs.shape}")
elif VERITE_VITL_IMG.exists():
    verite_img_embs = np.load(VERITE_VITL_IMG)
    verite_emb_dim  = verite_img_embs.shape[1]
    print(f"Loaded VERITE ViT-L/14 image embeddings: {verite_img_embs.shape}")
    # Text: encode on the fly with CLIP ViT-B/32
    print("Encoding VERITE captions with CLIP ViT-B/32 …")
    verite_txt_list = []
    batch_size = 128
    captions = df_verite['caption'].tolist()
    with torch.no_grad():
        for i in tqdm(range(0, len(captions), batch_size), desc='VERITE captions'):
            batch = captions[i:i+batch_size]
            tokens = clip.tokenize(batch, truncate=True).to(DEVICE)
            embs   = clip_model.encode_text(tokens).float().cpu().numpy()
            verite_txt_list.append(embs)
    verite_txt_embs = np.vstack(verite_txt_list)  # (N, 512)
    torch.cuda.empty_cache()
    # Pad/truncate image embeddings to match if needed (different dim)
    # We use only 512-d text; for image we compute CLIP B/32 from disk
    print(f"Text embs shape: {verite_txt_embs.shape}")
else:
    # Encode directly from images
    print("No pre-extracted VERITE embeddings found — encoding from images …")
    verite_img_list = []
    verite_txt_list = []
    batch_size = 64
    rows = df_verite.to_dict('records')

    for i in tqdm(range(0, len(rows), batch_size), desc='VERITE images'):
        batch = rows[i:i+batch_size]
        imgs, txts = [], []
        for r in batch:
            img_path = VERITE_IMGS / r['image_path']
            if not img_path.exists():
                # Try just the filename
                img_path = VERITE_IMGS / Path(r['image_path']).name
            try:
                img = clip_preprocess(Image.open(img_path).convert('RGB'))
            except Exception:
                img = clip_preprocess(Image.new('RGB', (224, 224)))
            imgs.append(img)
            txts.append(str(r['caption']))

        img_tensor = torch.stack(imgs).to(DEVICE)
        txt_tokens  = clip.tokenize(txts, truncate=True).to(DEVICE)
        with torch.no_grad():
            ie = clip_model.encode_image(img_tensor).float().cpu().numpy()
            te = clip_model.encode_text(txt_tokens).float().cpu().numpy()
        verite_img_list.append(ie)
        verite_txt_list.append(te)
        torch.cuda.empty_cache()

    verite_img_embs = np.vstack(verite_img_list)
    verite_txt_embs = np.vstack(verite_txt_list)
    verite_emb_dim  = verite_img_embs.shape[1]
    print(f"Encoded: img {verite_img_embs.shape}, txt {verite_txt_embs.shape}")

Loaded VERITE ViT-B/32 embeddings from cache: img (1001, 768), txt (1001, 768)


In [18]:
# ── Build VERITE feature dict ─────────────────────────────────────────────────
# label mapping: 'true' → 0, 'miscaptioned' → 1, 'out-of-context' → 1
LABEL_MAP = {'true': 0, 'miscaptioned': 1, 'out-of-context': 1}

def build_verite_features():
    results = {}
    for idx, row in tqdm(df_verite.iterrows(), total=len(df_verite), desc='VERITE'):
        label_str = str(row.get('label', row.get('gt_label', 'true'))).lower().strip()
        label = LABEL_MAP.get(label_str, 0)

        img_e = verite_img_embs[idx]
        txt_e = verite_txt_embs[idx] if verite_txt_embs is not None else np.zeros(512, dtype=np.float32)

        # Align dims: if img uses ViT-L/14 (768) and txt uses ViT-B/32 (512),
        # we store them separately; for clip_sim, use whichever pair matches
        if img_e.shape[0] == txt_e.shape[0]:
            clip_s = cosine_sim(img_e, txt_e)
        else:
            clip_s = 0.0  # mismatched dims — will be filled by ViT-B/32 pair later

        results[idx] = {
            'pair_key':      idx,
            'label':         label,
            'label_str':     label_str,
            'clip_img_emb':  l2_norm(img_e.astype(np.float32)),
            'clip_txt_emb':  l2_norm(txt_e.astype(np.float32)),
            'clip_sim':      np.float32(clip_s),
            'sbert_emb':     np.zeros(DIM_SBERT, dtype=np.float32),  # not available for VERITE
            'facenet_emb':   np.zeros(DIM_FACE,  dtype=np.float32),
            'places_emb':    np.zeros(DIM_PLACE, dtype=np.float32),
        }
    print(f"VERITE features built: {len(results)}")
    return results

verite_features = build_verite_features()

VERITE:   0%|          | 0/1001 [00:00<?, ?it/s]

VERITE features built: 1001


## 7. Build Stream B Feature Matrix — MMFakeBench

In [19]:
with open(MMFAKE_JSON) as f:
    mmfake_data = json.load(f)

# Accept list or dict
if isinstance(mmfake_data, dict):
    mmfake_pairs = list(mmfake_data.values()) if not isinstance(next(iter(mmfake_data.values())), (str, int, float)) else [{'key': k, **({'gt_answers': v} if isinstance(v, str) else v)} for k, v in mmfake_data.items()]
else:
    mmfake_pairs = mmfake_data

print(f"MMFakeBench test: {len(mmfake_pairs)} pairs")
print("Sample keys:", list(mmfake_pairs[0].keys()))

MMFakeBench test: 10000 pairs
Sample keys: ['text', 'image_path', 'text_source', 'image_source', 'gt_answers', 'fake_cls']


In [20]:
def mmfake_label(row):
    """Extract binary label (1=fake, 0=real) from gt_answers or label field."""
    gt = str(row.get('gt_answers', row.get('label', 'real'))).lower().strip()
    return 1 if 'fake' in gt else 0

def mmfake_img_path(row):
    """Resolve absolute image path for MMFakeBench."""
    ip = row.get('image_path', row.get('image', ''))
    p = Path(ip)
    # Try as-is
    if p.is_absolute() and p.exists():
        return p
    # Relative to project root
    candidate = ROOT / ip
    if candidate.exists():
        return candidate
    # Relative to MMFakeBench test root
    candidate2 = MMFAKE_ROOT / p.name
    if candidate2.exists():
        return candidate2
    # Walk subdirectories
    for sub in MMFAKE_ROOT.rglob(p.name):
        return sub
    return None

print("Encoding MMFakeBench images with CLIP ViT-B/32 …")
batch_size = 64
mm_img_embs, mm_txt_embs, mm_meta = [], [], []

for i in tqdm(range(0, len(mmfake_pairs), batch_size), desc='MMFake batches'):
    batch = mmfake_pairs[i:i+batch_size]
    imgs, txts, metas = [], [], []

    for j, row in enumerate(batch):
        img_p = mmfake_img_path(row)
        try:
            img = clip_preprocess(Image.open(img_p).convert('RGB')) if img_p else \
                  clip_preprocess(Image.new('RGB', (224, 224)))
        except Exception:
            img = clip_preprocess(Image.new('RGB', (224, 224)))
        imgs.append(img)
        txts.append(str(row.get('text', row.get('caption', ''))))
        metas.append({'idx': i + j, 'label': mmfake_label(row),
                      'image_path': str(img_p) if img_p else '',
                      'fake_cls': row.get('fake_cls', '')})

    img_tensor = torch.stack(imgs).to(DEVICE)
    txt_tokens  = clip.tokenize(txts, truncate=True).to(DEVICE)
    with torch.no_grad():
        ie = clip_model.encode_image(img_tensor).float().cpu().numpy()
        te = clip_model.encode_text(txt_tokens).float().cpu().numpy()
    mm_img_embs.append(ie)
    mm_txt_embs.append(te)
    mm_meta.extend(metas)
    torch.cuda.empty_cache()

mm_img_embs = np.vstack(mm_img_embs)
mm_txt_embs = np.vstack(mm_txt_embs)
print(f"MMFakeBench encoded: img {mm_img_embs.shape}, txt {mm_txt_embs.shape}")

Encoding MMFakeBench images with CLIP ViT-B/32 …


MMFake batches:   0%|          | 0/157 [00:00<?, ?it/s]

MMFakeBench encoded: img (10000, 512), txt (10000, 512)


In [21]:
def build_mmfake_features():
    results = {}
    for i, meta in enumerate(tqdm(mm_meta, desc='MMFake features')):
        img_e = mm_img_embs[i]
        txt_e = mm_txt_embs[i]
        clip_s = cosine_sim(img_e, txt_e)
        results[i] = {
            'pair_key':     i,
            'label':        meta['label'],
            'fake_cls':     meta['fake_cls'],
            'image_path':   meta['image_path'],
            'clip_img_emb': l2_norm(img_e.astype(np.float32)),
            'clip_txt_emb': l2_norm(txt_e.astype(np.float32)),
            'clip_sim':     np.float32(clip_s),
            'sbert_emb':    np.zeros(DIM_SBERT, dtype=np.float32),
            'facenet_emb':  np.zeros(DIM_FACE,  dtype=np.float32),
            'places_emb':   np.zeros(DIM_PLACE, dtype=np.float32),
        }
    print(f"MMFake features built: {len(results)}")
    return results

mmfake_features = build_mmfake_features()

MMFake features:   0%|          | 0/10000 [00:00<?, ?it/s]

MMFake features built: 10000


## 8. Save All Features

In [22]:
def save_pkl(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(obj, f, protocol=4)
    size_mb = path.stat().st_size / 1e6
    print(f"  Saved {path.name} ({size_mb:.1f} MB)")

print("Saving Stream B features …")
for split in ('train', 'val', 'test'):
    save_pkl(nc_features[split],
             STREAM_B_DIR / f'nc_{split}.pkl')

save_pkl(verite_features,  STREAM_B_DIR / 'verite.pkl')
save_pkl(mmfake_features,  STREAM_B_DIR / 'mmfake.pkl')

print("\n✅ All Stream B feature files saved to:", STREAM_B_DIR)

Saving Stream B features …
  Saved nc_train.pkl (1422.4 MB)
  Saved nc_val.pkl (140.6 MB)
  Saved nc_test.pkl (144.9 MB)
  Saved verite.pkl (19.7 MB)
  Saved mmfake.pkl (177.6 MB)

✅ All Stream B feature files saved to: f:\GitHub\eve-rag\cache\stream_b_features


## 9. Sanity Checks & Quick Stats

In [23]:
from sklearn.metrics import roc_auc_score

def quick_eval(features_dict, name):
    """Check class balance and clip_sim AUROC."""
    rows = list(features_dict.values())
    labels   = np.array([r['label'] for r in rows])
    clip_sim = np.array([r['clip_sim'] for r in rows])

    pos = labels.sum()
    neg = len(labels) - pos
    print(f"\n{name}: {len(labels)} pairs  |  pos={pos} ({100*pos/len(labels):.1f}%)  neg={neg}")
    print(f"  clip_sim mean(OOC)={clip_sim[labels==1].mean():.4f}  mean(genuine)={clip_sim[labels==0].mean():.4f}")

    if len(np.unique(labels)) == 2:
        # Lower clip_sim → more likely OOC (negate for AUROC)
        auc = roc_auc_score(labels, -clip_sim)
        print(f"  AUROC (clip_sim, negated): {auc:.4f}")

    # Embedding shapes
    sample = rows[0]
    for k in ('clip_img_emb', 'clip_txt_emb', 'sbert_emb', 'facenet_emb', 'places_emb'):
        print(f"  {k}: shape={sample[k].shape}")

for split in ('train', 'val', 'test'):
    quick_eval(nc_features[split], f'NC {split}')
quick_eval(verite_features, 'VERITE')
quick_eval(mmfake_features, 'MMFakeBench')


NC train: 71072 pairs  |  pos=35536 (50.0%)  neg=35536
  clip_sim mean(OOC)=0.2741  mean(genuine)=0.3067
  AUROC (clip_sim, negated): 0.7128
  clip_img_emb: shape=(512,)
  clip_txt_emb: shape=(512,)
  sbert_emb: shape=(768,)
  facenet_emb: shape=(2, 512)
  places_emb: shape=(2048,)

NC val: 7024 pairs  |  pos=3512 (50.0%)  neg=3512
  clip_sim mean(OOC)=0.2752  mean(genuine)=0.3071
  AUROC (clip_sim, negated): 0.7128
  clip_img_emb: shape=(512,)
  clip_txt_emb: shape=(512,)
  sbert_emb: shape=(768,)
  facenet_emb: shape=(512,)
  places_emb: shape=(1, 2048)

NC test: 7264 pairs  |  pos=3632 (50.0%)  neg=3632
  clip_sim mean(OOC)=0.2740  mean(genuine)=0.3072
  AUROC (clip_sim, negated): 0.7197
  clip_img_emb: shape=(512,)
  clip_txt_emb: shape=(512,)
  sbert_emb: shape=(768,)
  facenet_emb: shape=(512,)
  places_emb: shape=(1, 2048)

VERITE: 1001 pairs  |  pos=663 (66.2%)  neg=338
  clip_sim mean(OOC)=0.2666  mean(genuine)=0.3043
  AUROC (clip_sim, negated): 0.7093
  clip_img_emb: shape=

## 10. Summary

In [24]:
print("=" * 60)
print("STREAM B — clip_features.ipynb COMPLETE")
print("=" * 60)
print("\nSaved to cache/stream_b_features/:")
for f in sorted(STREAM_B_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s} {size_mb:6.1f} MB")
print("\nNext notebook: lora_stage1.ipynb")

STREAM B — clip_features.ipynb COMPLETE

Saved to cache/stream_b_features/:
  mmfake.pkl                                177.6 MB
  nc_test.pkl                               144.9 MB
  nc_train.pkl                             1422.4 MB
  nc_val.pkl                                140.6 MB
  verite.pkl                                 19.7 MB

Next notebook: lora_stage1.ipynb
